# C3 전체 데이터 최종 학습 및 제출 ZIP

C3의 동결 MERT·선형 분류층·출처 가중치·파일당 구간 기여량을 유지합니다. 공개 출처의 train/dev/sealed 전체를 최종 학습에 편입합니다. 이 실행에는 독립적인 최종 검증 세트가 없습니다.

제외: 정확히 동일한 파형 40개, 유효 구간이 없는 NSynth 10개, 부모 파일 ID가 없고 반복 타일링으로 만든 기존 추가 합성 80개. Echoes 저정보량 후보 2개는 유효 구간을 보존합니다. 파일 손상이나 오라벨로 확정한 목록이 아니라, 명시된 학습 품질 정책입니다.

최종 음악 학습 1,206개, 결합기 학습 1,886개. 음악 단독 음원으로 분류층을 학습하고, 음성·혼합·부분 위조 자료는 결합기에 사용합니다. 5-fold 원본 그룹 분리 예측으로 결합기를 학습한 뒤 음악 분류층은 전체 자료로 다시 학습합니다.


## 1. 환경과 입력 검증

In [1]:
from pathlib import Path
import csv, json, os, signal, subprocess, sys, time, zipfile
import numpy as np
import torch
ROOT = Path.cwd()
PROJECT = Path('/root/V6_L40S')
sys.path.insert(0, str(ROOT)); sys.path.insert(1, str(PROJECT))
from v6_data import sha256, write_json
from c3_full_train import load_records, train_final
PLAN = json.loads((ROOT / 'full_data_plan.json').read_text())
MANIFEST = json.loads((ROOT / 'input_manifest.json').read_text())
for name, digest in MANIFEST['sha256'].items():
    assert sha256(ROOT / name) == digest, name
RESUMED = float(os.environ['V7_RESUMED_EPOCH'])
DEADLINE = RESUMED + 1.5 * 3600
OUT = ROOT / 'final_run'; OUT.mkdir(exist_ok=True)
torch.set_num_threads(4)
assert time.time() < DEADLINE and torch.cuda.is_available()
print({'gpu': torch.cuda.get_device_name(), 'music_files': PLAN['music_files'],
       'fusion_files': PLAN['fusion_files'], 'evaluation': PLAN['evaluation']})


{'gpu': 'NVIDIA A100-SXM4-80GB', 'music_files': 1206, 'fusion_files': 1886, 'evaluation': 'No untouched holdout after full-data fit'}


## 2. 기존 캐시 재사용 및 새 입력 특징 추출
기존 train/dev는 검증된 C3/V6 캐시를 재사용합니다. 이번에 전체 학습에 추가하는 sealed의 고유 파일 190개만 새로 계산합니다.

In [2]:
records, engine = load_records(PLAN, PROJECT, DEADLINE)
write_json(OUT / 'feature_verification.json', {'files': len(records), 'asset_id': engine.asset_id,
    'cached_previous': sum(r['split'] != 'sealed' for r in PLAN['rows']),
    'new_sealed': sum(r['split'] == 'sealed' for r in PLAN['rows'])})
print('Verified features:', len(records))


/root/V6_L40S/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-15 03:39:14.536142006 [W:onnxruntime:, transformer_memcpy.cc:111 ApplyImpl] 24 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2026-09-15 03:39:14.571132656 [W:onnxruntime:, session_state.cc:1316 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2026-09-15 03:39:14.571164105 [W:onnxruntime:, session_state.cc:1318 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.
2026-09-15 03:39:15.439327721 [W:onnxruntime:Default, scatter_nd.h:51 ScatterNDWithAtomicReduction] ScatterND with reduction=='none' only guarantees to be correct if indices are not duplicate

Checkpoint path: /root/V6_L40S/model/assets/baseline/model/panns/Cnn14_mAP=0.431.pth


GPU number: 1


Verified features: 1886


## 3. 원본 그룹을 분리한 결합기 학습과 전체 재학습

C3: FMA 45%, NSynth 5%, 가짜 음악 50%(생성기별 균등), 파일당 같은 정규화 기여량과 epoch마다 하나의 유효 구간. AdamW lr 0.003, weight_decay 0.01, 300 epoch, seed 41.

파일·원본 곡·아티스트·부모 음원 그룹이 연결되면 같은 fold로 묶습니다. 각 파일의 결합기 입력은 그 원본을 학습하지 않은 음악 분류층으로 계산합니다. 전체 재학습 모델의 학습 점수를 검증 점수로 보고하지 않습니다.


In [3]:
metadata = train_final(PLAN, records, (OUT, engine.asset_id, DEADLINE))
print({key: metadata[key] for key in ['status','model_id','music_files','fusion_files','oof_folds','epochs_completed']})
print(metadata['final_evaluation'])
# Free the extraction models before fresh process-level packaging checks.
del engine
import gc
gc.collect(); torch.cuda.empty_cache()


{'status': 'complete', 'model_id': 'C3-full-clean-oof', 'music_files': 1206, 'fusion_files': 1886, 'oof_folds': 5, 'epochs_completed': 300}
Unavailable: all eligible public train/dev/sealed data used for final fitting


## 4. 오프라인 제출 ZIP 생성
이전 제출 의존성 오류를 막기 위해 NumPy·pandas와 torch·torchaudio·torchvision 버전을 고정한 패키저를 사용합니다. ZIP에 학습 음원은 넣지 않습니다.

In [4]:
from package_v6_final import package
ZIP = ROOT / 'C3_full_clean_submission.zip'
package_report = package(PROJECT / 'model/assets', OUT, ZIP)
print(package_report)


{'kind': 'submission', 'asset_id': 'b5da2bf48b57d9354312e532e579ec4765eb3c52fa335e10fc2f5845ad5b4f01', 'sha256': '70a4b9f917aebd471b1e9d5f05612d012cd9da52055b51cef09301a538159861', 'bytes': 6005189559, 'expanded_bytes': 6645859077, 'members': 42, 'trained': True, 'target_gpu_runtime': 'not verified by packaging'}


## 5. 실제 제출 진입점과 기준 실행 비교

음성 진짜·음성 가짜·혼합·부분 위조 4개 실제 파일을 사용합니다. 이는 실행·패키징 검증이며 독립 성능 평가가 아닙니다. ZIP에서 실행 코드와 학습 파라미터를 꺼내고, 자산은 원본 SHA256이 검증된 동일 파일을 연결합니다. 전체 ZIP CRC와 용량은 패키저에서 확인합니다. A100에서의 이 검사가 L4 전체 평가 60분 제한을 인증하지는 않습니다.


In [5]:
STAGE = OUT / 'packaged_smoke'; STAGE.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP) as archive:
    for item in archive.infolist():
        if item.filename == 'script.py' or item.filename.startswith(('model/runtime/', 'model/trained/')):
            target = (STAGE / item.filename).resolve()
            assert target.is_relative_to(STAGE.resolve())
            archive.extract(item, STAGE)
asset_link = STAGE / 'model/assets'
if not asset_link.exists(): asset_link.symlink_to(PROJECT / 'model/assets', target_is_directory=True)
INPUT = OUT / 'smoke_input'; INPUT.mkdir(exist_ok=True)
chosen = []
for panel, label in [('speech','0'),('speech','1'),('mixture','1'),('splice','1')]:
    row = next(r for r in PLAN['rows'] if r['panel'] == panel and r['FILE_FAKE'] == label)
    chosen.append(row)
    link = INPUT / (row['ID'] + '.wav')
    if not link.exists(): link.symlink_to(row['path'])
from v6_core import COLUMNS
sample = OUT / 'smoke_sample.csv'
with sample.open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=['ID', *COLUMNS], lineterminator='\n')
    writer.writeheader(); writer.writerows([dict(ID=r['ID'], **dict.fromkeys(COLUMNS, 0.)) for r in chosen])
seeded = "import random,numpy as np,torch,runpy,sys;random.seed(41);np.random.seed(41);torch.manual_seed(41);entry=sys.argv.pop(1);sys.argv[0]=entry;runpy.run_path(entry,run_name='__main__')"
common = ['--test-dir',str(INPUT),'--sample-submission',str(sample),'--device','cuda','--max-hours','0.2']
for entry, extra, name in [(ROOT / 'run_v6_final.py',['predict','--assets',str(PROJECT / 'model/assets'),'--work',str(OUT)],'reference'),
                            (STAGE / 'script.py',[],'packaged')]:
    command = [sys.executable,'-c',seeded,str(entry),*extra,*common,'--output',str(OUT / (name + '_predictions.csv'))]
    with (OUT / (name + '_inference.log')).open('w') as log:
        subprocess.run(command, cwd=ROOT, stdout=log, stderr=subprocess.STDOUT, check=True,
                       timeout=max(1, DEADLINE - time.time()))
def read_predictions(name):
    with (OUT / name).open() as stream: return list(csv.DictReader(stream))
a, b = read_predictions('reference_predictions.csv'), read_predictions('packaged_predictions.csv')
assert [r['ID'] for r in a] == [r['ID'] for r in b] == [r['ID'] for r in chosen]
x, y = (np.array([[float(r[k]) for k in COLUMNS] for r in values]) for values in (a,b))
assert np.isfinite(y).all() and (y >= 0).all() and (y <= 1).all()
np.testing.assert_allclose(x, y, atol=1e-5, rtol=0)
write_json(OUT / 'packaged_inference_check.json', {'real_files': len(chosen),
    'maximum_probability_difference': float(np.abs(x-y).max()), 'tolerance': 1e-5,
    'scope': 'Actual packaged CLI vs reference; assets are linked to identical verified files',
    'l4_full_evaluation_runtime': 'not verified'})
write_json(OUT / 'progress.json', {'stage': 'complete', 'submission': str(ZIP), 'utc': time.time()})
print('제출 ZIP과 실제 진입점 검증 완료. 결과를 다운로드하고 workspace를 Pause합니다.')


제출 ZIP과 실제 진입점 검증 완료. 결과를 다운로드하고 workspace를 Pause합니다.
